# Cloning ESeMan from git and preparing the executable


In [1]:
!git clone --recursive https://github.com/sayefsakin/eseman.git

Cloning into 'eseman'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 94 (delta 49), reused 64 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 138.64 KiB | 5.54 MiB/s, done.
Resolving deltas: 100% (49/49), done.
Submodule 'hclust-cpp' (https://github.com/sayefsakin/hclust-cpp.git) registered for path 'hclust-cpp'
Submodule 'rapidjson' (https://github.com/Tencent/rapidjson.git) registered for path 'rapidjson'
Cloning into '/content/eseman/hclust-cpp'...
remote: Enumerating objects: 68, done.        
remote: Counting objects: 100% (68/68), done.        
remote: Compressing objects: 100% (47/47), done.        
remote: Total 68 (delta 34), reused 47 (delta 18), pack-reused 0 (from 0)        
Receiving objects: 100% (68/68), 35.35 KiB | 2.95 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Cloning into '/content/eseman/rapidjson'...
remote: Enumerating objects: 25114,

# Install the dependencies

In [2]:
%%shell
apt-get install libboost-all-dev
apt-get install liblmdb-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libboost-all-dev is already the newest version (1.74.0.3ubuntu7).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  lmdb-doc
The following NEW packages will be installed:
  liblmdb-dev lmdb-doc
0 upgraded, 2 newly installed, 0 to remove and 38 not upgraded.
Need to get 340 kB of archives.
After this operation, 2,592 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 liblmdb-dev amd64 0.9.24-1build2 [63.7 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 lmdb-doc all 0.9.24-1build2 [276 kB]
Fetched 340 kB in 0s (1,729 kB/s)
Selecting previously unselected package liblmdb-dev:amd64.
(Reading database ... 126675 files and directories currently installed.)
Preparing to unpack

In [3]:
%cd eseman/

/content/eseman


In [4]:
!make

g++ -Wall -g -DNO_INCLUDE_FENV  -c hclust-cpp/fastcluster.cpp
rm -f eseman_data_server
g++ -g -Wall -Ofast -Wno-stringop-overflow -march=native -mtune=native -flto=auto -funroll-loops -fno-plt  -o eseman_data_server eseman_data_server.cpp agglomerate_clustering.cpp eseman_kdt.cpp fastcluster.o -I./rapidjson/include -llmdb


In [5]:
!./eseman_data_server -h

Usage: ./eseman_data_server [options]

Options:
   -h, --help     Show this help message.
   -s, --start    Start the server.
   -b, --bundle   Bundle the input file and store into LMDB.
   -p, --port    <PORT>     Server port (default: 8080).
   -i, --input   <FILE>     Specify the event sequence input file location. The input file should be in JSON format.
   -m, --model   <MODEL>    Specify the data structure, AGC for agglomerative clustering, KDT for KD-Tree. (default: KDT).



# Bundle input file

In [6]:
!./eseman_data_server -b -i input_data/ecc21d0a-112a-4b52-8cdd-6aca80adde93.json

# Start the server

In [7]:
!nohup ./eseman_data_server -s -p 5500 &

nohup: appending output to 'nohup.out'


# Test if the server is running

In [8]:
!curl http://127.0.0.1:5500/get-data-in-range?bins=1000

{"data":[{"track":"1","utils":["0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.500000","0.500000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","0.000000","

# An example using Vega

In [9]:
vega_spec = {
  "$schema": "https://vega.github.io/schema/vega/v5.json",
  "description": "Gantt with bins slider (dynamic URL fetch)",
  "width": 600,
  "height": 200,
  "padding": 5,

  "signals": [
    {
        "name": "bins", "value": 1000,
        "bind": {"input": "range", "min": 100, "max": 2000, "step": 100, "debounce": 300, "name": "Bins: "}
    },
    {"name": "apiUrl", "value": "http://localhost:5500/get-data-in-range"},
    {
      "name": "begin",
      "value": 36546573,
      "bind": { "input": "range", "min": 36546573, "max": 372949710, "step": 1000000, "debounce": 50, "name": "Begin: " },
      "on": [
        { "events": "data:meta", "update": "data('meta').length ? data('meta')[0].metadata.begin : begin" }
      ]
    },
    {
      "name": "end",
      "value": 372949710,
      "bind": { "input": "range", "min": 36546573, "max": 372949710, "step": 1000000, "debounce": 50, "name": "End: " },
      "on": [
        { "events": "data:meta", "update": "data('meta').length ? data('meta')[0].metadata.end : end" }
      ]
    }
  ],

  "data": [
    {
      "name": "raw",
      # "url": {"signal": "apiUrl + '?bins=' + bins"},
      "url": {"signal": "apiUrl + '?bins=' + bins + '&begin=' + begin + '&end=' + end"},
      "format": {"type": "json"}
    },
    {
      "name": "rows",
      "source": "raw",
      "transform": [
        {"type": "formula", "expr": "datum.metadata.begin", "as": "metaBegin"},
        {"type": "formula", "expr": "datum.metadata.end", "as": "metaEnd"},
        {"type": "formula", "expr": "datum.metadata.bins", "as": "metaBins"},
        {"type": "flatten", "fields": ["data"], "as": ["row"]},
        {"type": "formula", "expr": "datum.row.track", "as": "track"},
        {"type": "flatten", "fields": ["row.utils"], "as": ["utilFloat"], "index": "binIndex"},
        {"type": "formula", "expr": "toNumber(datum.utilFloat)", "as": "utilFloat"},
        {"type": "filter", "expr": "datum.utilFloat > 0"},
        {"type": "formula", "expr": "(datum.metaEnd - datum.metaBegin)/datum.metaBins", "as": "binDuration"},
        {"type": "formula", "expr": "datum.metaBegin + (datum.binIndex * datum.binDuration)", "as": "timeStart"},
        {"type": "formula", "expr": "datum.timeStart + datum.binDuration", "as": "timeEnd"},
        {"type": "formula", "expr": "datum.utilFloat >= 1 ? 'solid' : 'partial'", "as": "barType"}
      ]
    }
  ],

  "scales": [
    {"name": "x", "type": "utc", "domain": {"data": "rows", "fields": ["timeStart", "timeEnd"]}, "range": "width", "nice": False},
    {"name": "y", "type": "band", "domain": {"data": "rows", "field": "track"}, "range": "height", "padding": 0.1},
    {"name": "color", "type": "ordinal", "domain": ["solid", "partial"], "range": ["#000000", "#9ca3af"]}
  ],

  "axes": [
    {"orient": "bottom", "scale": "x", "title": "Time","format": "%s","labelOverlap": "greedy"},
    {"orient": "left", "scale": "y", "title": "Tracks", "grid": True,"bandPosition": 0}
  ],

  "marks": [
    {
      "type": "rect",
      "from": {"data": "rows"},
      "encode": {
        "enter": {
          "x": {"scale": "x", "field": "timeStart"},
          "x2": {"scale": "x", "field": "timeEnd"},
          "y": {"scale": "y", "field": "track"},
          "height": {"signal": "bandwidth('y')"},
          "fill": {"scale": "color", "field": "barType"},
          "tooltip": {"signal": "{track: datum.track, util: datum.utilFloat}"}
        }
      }
    }
  ]
}


In [10]:
from IPython.display import HTML
import json, uuid

def vega(spec: dict):
    """Render a Vega spec inline using vega-embed (method 1)."""
    div_id = f"vis_{uuid.uuid4().hex}"
    html = f"""
<div id="{div_id}"></div>
<script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
<script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
<script>
  const spec = {json.dumps(spec)};
  vegaEmbed("#{div_id}", spec, {{actions: false}}).catch(console.error);
</script>
"""
    return HTML(html)

vega(vega_spec)

# Stop the server

In [9]:
%%shell
ESEMANPID=`ps aux | grep "./eseman_data_server"  | awk '{print $2}' | head -n 1`
echo $ESEMANPID
kill $ESEMANPID

56409
